
# Convolutional Autoencoder for Sparse Time-of-Flight Sensor Data Compression

## Overview

This notebook demonstrates the development and training of a convolutional autoencoder specifically designed to handle **extremely sparse** Time-of-Flight (ToF) sensor data. The key insight driving this approach is that traditional normalization techniques fail catastrophically on sparse data, making autoencoder-based compression not just beneficial, but essential.

## The Sparsity Problem

### Why Traditional Normalization Fails

Time-of-Flight sensors generate highly sparse data where:
- **The vast majority of values are zeros** (empty space measurements)
- **A very small minority of pixels contain actual distance readings**
- **StandardScaler becomes unstable** with sparse data

```python
# Example of sparse ToF data:
sparse_tof_sample = [
    [0, 0, 0, 15.2, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 12.8, 0, 0], 
    [0, 0, 0, 0, 0, 0, 0, 0],
    # ... mostly zeros with occasional distance readings
]

# Traditional normalization disaster:
mean = sparse_data.mean()  # ≈ 0 (dominated by zeros)
std = sparse_data.std()    # Very small (mostly zeros)
normalized = (data - 0) / tiny_std  # Non-zero values explode!
```


In [ ]:

### The Core Challenge

```python
# Memory and computational issues:
Original_ToF_Shape = (batch_size, 320)  # 5 sensors × 8×8 each, 95% sparse
Memory_Usage = "Huge for mostly empty data"
Normalization_Result = "Catastrophic value explosion"
NN_Training = "Poor gradients due to extreme value ranges"
```

## Our Autoencoder Solution

### Why Autoencoders Excel at Sparse Data

1. **Learned Compression**: Discovers meaningful patterns in sparse spatial arrangements
2. **Implicit Normalization**: Compression/reconstruction acts as learned normalization
3. **Spatial Awareness**: Preserves spatial relationships between sparse measurements
4. **Noise Filtering**: Naturally filters sensor noise while preserving signal

## Data Structure & Preprocessing

### Original ToF Data Format
- **5 sensor arrays** per timestamp (tof_1_v0 through tof_5_v63)
- **8×8 spatial resolution** per sensor → 64 values per sensor
- **Total: 320 values per timestamp**, mostly zeros
- **Spatial padding** to 18×18 for cleaner convolution math

```python
# Reshaping sparse ToF data
ADDED_PADDING = 4
reshape_tof_row = lambda tof_row: torch.reshape(
    F.pad(tof_row, (0, ADDED_PADDING)), (18, 18)
)

# Data preprocessing for sparse data
def preprocess_sparse_tof(tof_tensor, data_mean, data_std):
    # Clamp to avoid negative distances
    clamped = torch.clamp(tof_tensor, min=-1e-6)
    # Sigmoid normalization works better than StandardScaler for sparse data
    return torch.sigmoid((clamped - data_mean) / data_std)
```

## Architecture Design

### Encoder-Decoder Architecture

```python
# Autoencoder specifically designed for sparse spatial data
model = nn.Sequential(
    # Encoder: Progressive spatial compression
    nn.Conv2d(1, 8, kernel_size=3, stride=2, padding=1),    # 18×18 → 9×9
    GeneralRelu(leak=0.4, sub=0.1),
    nn.BatchNorm2d(8),
    
    nn.Conv2d(8, 16, kernel_size=3, stride=2, padding=1),   # 9×9 → 5×5  
    GeneralRelu(leak=0.4, sub=0.1),
    nn.BatchNorm2d(16),
    
    # Decoder: Spatial reconstruction
    reverse_conv(16, 8),                                     # 5×5 → 10×10
    GeneralRelu(leak=0.4, sub=0.1),
    nn.BatchNorm2d(8),
    
    reverse_conv(8, 1),                                      # 10×10 → 20×20
    nn.ZeroPad2d(-1)                                        # 20×20 → 18×18
)
```

### Key Design Decisions for Sparse Data

1. **Progressive Channel Expansion**: 1 → 8 → 16 → 8 → 1
   - Learns hierarchical sparse patterns
   - Bottleneck at 5×5×16 = 400 compressed values

2. **GeneralRelu with Leak**: Handles sparse gradients better than standard ReLU

3. **BatchNorm after each layer**: Stabilizes training with sparse inputs

4. **Symmetric Architecture**: Ensures reconstruction fidelity

## Training Process & Results

### Training Configuration
```python
# Optimized for sparse data learning
optimizer = AdamW(lr=1e-4)
loss_function = F.mse_loss  # Reconstruction quality
epochs = 5  # Rapid convergence due to sparse patterns
```

### Training Results: Excellent Sparse Data Compression

![Training Progress](path/to/training_results.png)

*Figure 1: Training curves showing rapid convergence from MSE ~0.289 to ~0.041 in just 5 epochs*

| Epoch | Train Loss | Valid Loss | Train MSE | Valid MSE |
|-------|------------|------------|-----------|-----------|
| 0     | 0.530      | 0.762      | 0.289     | 0.514     |
| 1     | 0.306      | 0.384      | 0.170     | 0.249     |
| 2     | 0.183      | 0.203      | 0.099     | 0.120     |
| 3     | 0.122      | 0.123      | 0.062     | 0.064     |
| 4     | 0.093      | 0.090      | 0.045     | 0.041     |

### Why This Works So Well

**4% final reconstruction error** on extremely sparse data is exceptional because:

1. **Sparse Pattern Learning**: Autoencoder learned where objects typically appear
2. **Background Subtraction**: Automatically handles "empty space" efficiently  
3. **Spatial Relationship Preservation**: Maintains gesture-relevant spatial patterns
4. **Noise Filtering**: Filters sensor noise while preserving meaningful signals

## Feature Extraction Strategy

### Encoder Output: 5×5×16 Feature Maps

The encoder produces rich spatial features that need careful handling:

```python
# Original approach - too much information loss
encoder_with_pooling = nn.Sequential(
    encoder,  # → (batch, 16, 5, 5)
    nn.AdaptiveAvgPool2d((1,1))  # → (batch, 16, 1, 1)
)
# Result: 400 → 16 features (massive information loss)